In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
df = pd.read_parquet("../data/aircraft engine/PM_train.parquet")
df

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,max,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,192,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,192,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,192,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,192,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,192,187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735,200,4
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594,200,3
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333,200,2
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640,200,1


In [3]:
x = df.drop(['id','cycle','RUL', 'max'], axis=1)
y = df['RUL']
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.21, random_state=42)
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [4]:
import json
with open('note.json', 'r') as file:
    info = json.loads(file.read())

In [5]:
info

{'gb_model': {'r2_score': 0.8666493068005445,
  'best_params': {'subsample': 1.0,
   'n_estimators': 2000,
   'min_samples_split': 5,
   'min_samples_leaf': 2,
   'max_depth': 5,
   'loss': 'squared_error',
   'learning_rate': 0.1,
   'criterion': 'friedman_mse',
   'alpha': 0.5}},
 'gb_model_2': {'r2_score': 0.8653454963351543,
  'best_params': {'alpha': 0.7,
   'criterion': 'friedman_mse',
   'learning_rate': 0.07536840425014499,
   'loss': 'squared_error',
   'max_depth': 9,
   'min_samples_leaf': 3,
   'min_samples_split': 7,
   'n_estimators': 2000,
   'subsample': 0.8}},
 'xgb_model_2': {'r2_score': 0.633815586566925,
  'best_params': {'colsample_bytree': 1.0,
   'gamma': 0.1,
   'grow_policy': 'depthwise',
   'learning_rate': 0.010724528148632458,
   'max_depth': None,
   'n_estimators': 2000,
   'reg_alpha': 0.1,
   'reg_lambda': 0.1,
   'subsample': 0.8}}}

In [6]:
xgb_model = xgb.XGBRegressor(colsample_bytree=1, learning_rate=0.01, max_depth=None, n_estimators=2000, reg_alpha=0.1, gamma=0.1, reg_lambda=10, subsample=0.8, random_state=42)

In [7]:
xgb_model.fit(x_train_scaled, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None, colsample_bytree=1,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.01, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [8]:
y_pred = xgb_model.predict(x_test_scaled)

In [9]:
r2_score(y_test, y_pred)

0.6233607530593872

In [10]:
test_df = pd.read_csv("../data/aircraft engine/PM_test.csv")
test_df
# g = test_df.groupby('id')['cycle'].max()
# g

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13091,100,194,0.0049,0.0000,100.0,518.67,643.24,1599.45,1415.79,14.62,...,520.69,2388.00,8213.28,8.4715,0.03,394,2388,100.0,38.65,23.1974
13092,100,195,-0.0011,-0.0001,100.0,518.67,643.22,1595.69,1422.05,14.62,...,521.05,2388.09,8210.85,8.4512,0.03,395,2388,100.0,38.57,23.2771
13093,100,196,-0.0006,-0.0003,100.0,518.67,643.44,1593.15,1406.82,14.62,...,521.18,2388.04,8217.24,8.4569,0.03,395,2388,100.0,38.62,23.2051
13094,100,197,-0.0038,0.0001,100.0,518.67,643.26,1594.99,1419.36,14.62,...,521.33,2388.08,8220.48,8.4711,0.03,395,2388,100.0,38.66,23.2699


In [11]:
test = test_df.drop(['id','cycle'], axis=1)
test_id = test_df['id']
test = scaler.transform(test)

In [12]:
truth_df = pd.read_csv("../data/aircraft engine/PM_truth.csv")
truth_df

,id,cycle
0,1,112
1,2,98
2,3,69
3,4,82
4,5,91
...,...,...
95,96,137
96,97,82
97,98,59
98,99,117


In [13]:
test_df['id'] = test_id
test_df['pred'] = xgb_model.predict(test)

In [14]:
test_df[test_df.id==2][:5]

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,pred
31,2,1,-0.0009,0.0004,100.0,518.67,642.66,1589.30,1407.16,14.62,...,2388.14,8129.59,8.4283,0.03,392,2388,100.0,39.00,23.3923,162.589905
32,2,2,-0.0011,0.0002,100.0,518.67,642.51,1588.43,1405.47,14.62,...,2388.08,8120.05,8.4414,0.03,393,2388,100.0,38.84,23.2902,151.922791
33,2,3,0.0002,0.0003,100.0,518.67,642.58,1595.60,1410.86,14.62,...,2388.08,8126.75,8.3804,0.03,394,2388,100.0,39.02,23.4064,148.764847
34,2,4,0.0025,0.0001,100.0,518.67,642.31,1583.43,1408.23,14.62,...,2388.06,8129.91,8.4342,0.03,393,2388,100.0,38.82,23.4699,178.687958
35,2,5,0.0004,-0.0004,100.0,518.67,642.77,1585.03,1407.60,14.62,...,2388.11,8127.01,8.4247,0.03,392,2388,100.0,38.81,23.3895,158.808853


In [15]:
test_df_group_mean = test_df.groupby('id')['pred'].mean()
test_df_group_mean

id
1      175.037964
2      149.091202
3      122.510941
4      137.928284
5      136.763138
          ...    
96     165.681015
97     139.282318
98     133.011246
99     178.660477
100    127.517960
Name: pred, Length: 100, dtype: float32

In [16]:
test_df_group_count = test_df.groupby('id').count()
test_df_group_count

,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,s6,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,pred
id,,,,,,,,,,,,,,,,,,,,,
1,31,31,31,31,31,31,31,31,31,31,...,31,31,31,31,31,31,31,31,31,31
2,49,49,49,49,49,49,49,49,49,49,...,49,49,49,49,49,49,49,49,49,49
3,126,126,126,126,126,126,126,126,126,126,...,126,126,126,126,126,126,126,126,126,126
4,106,106,106,106,106,106,106,106,106,106,...,106,106,106,106,106,106,106,106,106,106
5,98,98,98,98,98,98,98,98,98,98,...,98,98,98,98,98,98,98,98,98,98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,97,97,97,97,97,97,97,97,97,97,...,97,97,97,97,97,97,97,97,97,97
97,134,134,134,134,134,134,134,134,134,134,...,134,134,134,134,134,134,134,134,134,134
98,121,121,121,121,121,121,121,121,121,121,...,121,121,121,121,121,121,121,121,121,121


In [17]:
test_df_group_max = test_df.groupby('id')['pred'].max()
test_df_group_max

id
1      212.366837
2      178.687958
3      164.066101
4      172.775589
5      180.060043
          ...    
96     211.016312
97     203.164383
98     200.288162
99     226.352036
100    216.809525
Name: pred, Length: 100, dtype: float32

In [18]:
test_df_group_min = test_df.groupby('id')['pred'].min()
test_df_group_min

id
1      144.576645
2      108.172806
3       51.761192
4       89.130836
5       80.404289
          ...    
96     111.054779
97      68.497635
98      72.859978
99     138.097336
100     15.006837
Name: pred, Length: 100, dtype: float32

In [19]:
test_df_sorted = test_df.sort_values(['id', 'pred'], ascending=True)

In [20]:
test_df_sorted[test_df_sorted.id==2][:5]['pred'].mean()

124.5486

In [21]:
test_df_sorted[test_df_sorted.id==1][:5]['pred'].mean()

154.57254

In [22]:
test_df_sorted['avg_pred'] = test_df_sorted['id'].map(lambda x : test_df_sorted[test_df_sorted.id==x][:5]['pred'].mean())

In [23]:
test_df_sorted

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,pred,avg_pred
24,1,25,0.0028,-0.0003,100.0,518.67,642.25,1582.43,1400.23,14.62,...,8128.65,8.4007,0.03,393,2388,100.0,38.96,23.3785,144.576645,154.572540
30,1,31,-0.0006,0.0004,100.0,518.67,642.58,1581.22,1398.91,14.62,...,8130.11,8.4024,0.03,393,2388,100.0,38.81,23.3552,154.799881,154.572540
15,1,16,-0.0018,0.0003,100.0,518.67,642.32,1584.51,1407.76,14.62,...,8133.83,8.4300,0.03,390,2388,100.0,38.87,23.3484,156.913315,154.572540
19,1,20,0.0011,-0.0001,100.0,518.67,642.61,1587.78,1400.70,14.62,...,8128.59,8.4099,0.03,392,2388,100.0,39.00,23.3325,157.931778,154.572540
29,1,30,-0.0025,0.0004,100.0,518.67,642.79,1585.72,1400.97,14.62,...,8134.79,8.4110,0.03,391,2388,100.0,39.09,23.4069,158.641083,154.572540
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12902,100,5,-0.0011,0.0005,100.0,518.67,642.01,1589.70,1397.19,14.62,...,8130.34,8.4237,0.03,391,2388,100.0,38.97,23.4710,207.338516,23.049793
12936,100,39,-0.0002,0.0005,100.0,518.67,642.20,1582.30,1397.95,14.62,...,8138.45,8.3770,0.03,391,2388,100.0,38.92,23.4326,207.371033,23.049793
12921,100,24,0.0006,0.0002,100.0,518.67,642.63,1582.93,1392.38,14.62,...,8132.74,8.4041,0.03,390,2388,100.0,39.00,23.4300,213.522720,23.049793
12937,100,40,0.0016,0.0004,100.0,518.67,641.92,1580.46,1397.67,14.62,...,8133.67,8.4099,0.03,393,2388,100.0,39.15,23.4710,215.436661,23.049793


In [24]:
pred_cycle = test_df_sorted.groupby('id', as_index=False)['avg_pred'].mean()

In [25]:
pred_cycle['avg_pred']

0     154.572540
1     124.548599
2      62.764355
3      96.890228
4      96.765396
         ...    
95    129.715240
96     83.714371
97     84.085617
98    148.556366
99     23.049793
Name: avg_pred, Length: 100, dtype: float32

In [26]:
truth_df['pred_cycle'] = pred_cycle['avg_pred']

In [27]:
truth_df

,id,cycle,pred_cycle
0,1,112,154.572540
1,2,98,124.548599
2,3,69,62.764355
3,4,82,96.890228
4,5,91,96.765396
...,...,...,...
95,96,137,129.715240
96,97,82,83.714371
97,98,59,84.085617
98,99,117,148.556366


In [28]:
truth_df['diff'] = truth_df['cycle'] - truth_df['pred_cycle']

In [29]:
truth_df[(10<truth_df['diff']) | (-10>truth_df['diff'])].count()

id            53
cycle         53
pred_cycle    53
diff          53
dtype: int64

In [32]:
r2_score(truth_df['cycle'], truth_df['pred_cycle'])

0.6369857788085938